In [0]:
# Databricks widgets for portable configuration
dbutils.widgets.text("catalog", "workspace", "Catalog Name")
dbutils.widgets.text("schema_bronze", "cbl_bronze", "Bronze Schema")
dbutils.widgets.text("schema_silver", "cbl_silver", "Silver Schema")
dbutils.widgets.text("schema_gold", "cbl_gold", "Gold Schema")

# Read widget values into variables
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

# Print configuration for verification
print("Configuration:")
print(f"  Catalog: {catalog}")
print(f"  Bronze Schema: {schema_bronze}")
print(f"  Silver Schema: {schema_silver}")
print(f"  Gold Schema: {schema_gold}")
print(f"\nVolume path: /Volumes/{catalog}/{schema_bronze}/raw/")

In [0]:
# Create bronze, silver, and gold schemas using variables
print("Creating schemas and volume...\n")

# Create schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_bronze}")
print(f"✓ Schema {catalog}.{schema_bronze} ready")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_silver}")
print(f"✓ Schema {catalog}.{schema_silver} ready")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_gold}")
print(f"✓ Schema {catalog}.{schema_gold} ready")

# Create raw volume in bronze schema
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_bronze}.raw")
print(f"✓ Volume {catalog}.{schema_bronze}.raw ready")

In [0]:
import zipfile
import os

# Extract the main zip file
zip_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1-20260807.zip"
extract_to = f"/Volumes/{catalog}/{schema_bronze}/raw/"

print("Extracting main zip file...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

# Extract the nested zip file
nested_zip = os.path.join(extract_to, "Data_Module_1/lab_landing_zone.zip")
print("Extracting nested zip file...")
with zipfile.ZipFile(nested_zip, 'r') as zip_ref:
    zip_ref.extractall(os.path.join(extract_to, "Data_Module_1"))

# List top-level folders in Data_Module_1
base_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1"
print(f"\nTop-level folders in {base_path}:")
for item in sorted(os.listdir(base_path)):
    item_path = os.path.join(base_path, item)
    if os.path.isdir(item_path):
        print(f"  [DIR] {item}")

# List folders inside the landing directory
landing_path = os.path.join(base_path, "landing")
if os.path.exists(landing_path):
    print(f"\nTop-level folders in landing directory:")
    for item in sorted(os.listdir(landing_path)):
        item_path = os.path.join(landing_path, item)
        if os.path.isdir(item_path):
            print(f"  [DIR] {item}")

In [0]:
import os

landing_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing"

print("Child folders in each landing directory folder:\n")
for parent_folder in sorted(os.listdir(landing_path)):
    parent_path = os.path.join(landing_path, parent_folder)
    if os.path.isdir(parent_path):
        print(f"[{parent_folder}]")
        child_folders = [item for item in os.listdir(parent_path) if os.path.isdir(os.path.join(parent_path, item))]
        if child_folders:
            for child in sorted(child_folders):
                print(f"  └─ {child}")
        else:
            print("  └─ (no subfolders)")
        print()

In [0]:
import os

base_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing"

# Folders to check
folders_to_check = [
    "sap_s4/sd_billing",
    "sfa_app/orders",
    "external_api/weather"
]

print("Entry counts in specified folders:\n")
for folder in folders_to_check:
    full_path = os.path.join(base_path, folder)
    if os.path.exists(full_path):
        entries = os.listdir(full_path)
        count = len(entries)
        print(f"{folder}: {count} entries")
    else:
        print(f"{folder}: (folder not found)")

In [0]:
import os
import gzip

dir_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing/sap_s4/sd_billing/ingest_date=2024-03-01/"

# List files in the directory
files = [f for f in os.listdir(dir_path) if f.endswith('.gz') or f.endswith('.csv')]

if files:
    # Get the first file
    filename = files[0]
    file_path = os.path.join(dir_path, filename)
    
    print(f"Filename: {filename}\n")
    print("First 3 lines:\n")
    
    # Read first 3 lines
    if filename.endswith('.gz'):
        with gzip.open(file_path, 'rt') as f:
            for i, line in enumerate(f):
                if i < 3:
                    print(line.rstrip())
                else:
                    break
    else:
        with open(file_path, 'r') as f:
            for i, line in enumerate(f):
                if i < 3:
                    print(line.rstrip())
                else:
                    break
else:
    print("No CSV or gzipped files found in the directory")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
import uuid

# Generate a unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())

# Define paths
source_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing/sap_s4/sd_billing/*/*"
target_table = f"{catalog}.{schema_bronze}.sap_billing"

print(f"Starting batch ingestion...")
print(f"Source: {source_path}")
print(f"Target: {target_table}")
print(f"Batch ID: {batch_id}\n")

# Read all CSV files with batch loading
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("delimiter", "|")
    .option("inferSchema", "false")  # No type inference - all columns as STRING
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")  # Capture malformed rows
    .load(source_path)
)

# Add lineage metadata columns
df_with_metadata = (df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

# Write to bronze table with overwrite mode
df_with_metadata.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

row_count = df_with_metadata.count()
print(f"\nIngestion completed successfully!")
print(f"Data written to: {target_table}")
print(f"Rows ingested: {row_count:,}")

In [0]:
# Check the ingested data
query = f"""
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT _batch_id) as batch_count,
  COUNT(DISTINCT DATE(_ingest_timestamp)) as date_partitions,
  COUNT(ZZLOYALTY_ID) as rows_with_loyalty_id,
  COUNT(ZZPROMO_CODE) as rows_with_promo_code
FROM {catalog}.{schema_bronze}.sap_billing
"""

result = spark.sql(query)
display(result)

In [0]:
# Sample data showing all columns including metadata
query = f"""
SELECT 
  MANDT, VBELN, POSNR, FKART, FKDAT,
  ZZLOYALTY_ID, ZZPROMO_CODE,
  _source_file,
  _ingest_timestamp,
  _batch_id
FROM {catalog}.{schema_bronze}.sap_billing
WHERE ZZLOYALTY_ID IS NOT NULL
LIMIT 3
"""

result = spark.sql(query)
display(result)

In [0]:
import os

masterdata_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing/sap_s4/masterdata/"

print("Files in masterdata folder:\n")
for file in sorted(os.listdir(masterdata_path)):
    file_path = os.path.join(masterdata_path, file)
    if os.path.isfile(file_path):
        file_size = os.path.getsize(file_path)
        print(f"  {file} ({file_size:,} bytes)")

In [0]:
# Drop the incorrectly created tables
print("Dropping tables if they exist...")

spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema_bronze}.kna1_customer")
print(f"  Dropped {catalog}.{schema_bronze}.kna1_customer")

spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema_bronze}.mara_material")
print(f"  Dropped {catalog}.{schema_bronze}.mara_material")

In [0]:
# Read first line of both CSV files
files = [
    f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing/sap_s4/masterdata/KNA1_customer_extract.csv",
    f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing/sap_s4/masterdata/MARA_material_extract.csv"
]

for file_path in files:
    print(f"File: {file_path.split('/')[-1]}")
    with open(file_path, 'r') as f:
        first_line = f.readline().strip()
        print(f"First line: {first_line}")
        print(f"Length: {len(first_line)}")
        print()

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import StringType
import uuid

# Generate a unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())

# Define paths
source_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing/sap_s4/masterdata/KNA1_customer_extract.csv"
target_table = f"{catalog}.{schema_bronze}.kna1_customer"

print(f"Starting ingestion for KNA1 customer...")
print(f"Source: {source_path}")
print(f"Target: {target_table}")
print(f"Batch ID: {batch_id}\n")

# Read CSV with all columns as string
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("delimiter", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .load(source_path)
)

# Add lineage metadata columns
df_with_metadata = (df
    .withColumn("_source_file", lit(source_path))
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

# Write to bronze table with append mode and column mapping enabled
df_with_metadata.write \
    .format("delta") \
    .mode("append") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(target_table)

row_count = df_with_metadata.count()
print(f"\nIngestion completed successfully!")
print(f"Data written to: {target_table}")
print(f"Rows ingested: {row_count:,}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import StringType
import uuid

# Generate a unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())

# Define paths
source_path = f"/Volumes/{catalog}/{schema_bronze}/raw/Data_Module_1/landing/sap_s4/masterdata/MARA_material_extract.csv"
target_table = f"{catalog}.{schema_bronze}.mara_material"

print(f"Starting ingestion for MARA material...")
print(f"Source: {source_path}")
print(f"Target: {target_table}")
print(f"Batch ID: {batch_id}\n")

# Read CSV with all columns as string
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("delimiter", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .load(source_path)
)

# Add lineage metadata columns
df_with_metadata = (df
    .withColumn("_source_file", lit(source_path))
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

# Write to bronze table with append mode and column mapping enabled
df_with_metadata.write \
    .format("delta") \
    .mode("append") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(target_table)

row_count = df_with_metadata.count()
print(f"\nIngestion completed successfully!")
print(f"Data written to: {target_table}")
print(f"Rows ingested: {row_count:,}")

In [0]:
# Check customer table schema
result = spark.sql(f"DESCRIBE TABLE {catalog}.{schema_bronze}.kna1_customer")
display(result)

In [0]:
%sql
-- Summary of all bronze tables
SELECT 
  'sap_billing' as table_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT _batch_id) as batch_count
FROM workspace.cbl_bronze.sap_billing

UNION ALL

SELECT 
  'kna1_customer' as table_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT _batch_id) as batch_count
FROM workspace.cbl_bronze.kna1_customer

UNION ALL

SELECT 
  'mara_material' as table_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT _batch_id) as batch_count
FROM workspace.cbl_bronze.mara_material

ORDER BY table_name;

In [0]:
# Define catalog and schema variables
catalog = "workspace"
schema_bronze = "cbl_bronze"
schema_silver = "cbl_silver"
schema_gold = "cbl_gold"

# Construct table name
table_name = f"{catalog}.{schema_bronze}.sap_billing"

print(f"Reading from: {table_name}\n")

# Read the table
df_billing = spark.table(table_name)

print(f"Total rows: {df_billing.count():,}")
print(f"Columns: {len(df_billing.columns)}")

# Show sample
display(df_billing.limit(5))

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
import uuid

# Generate a unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())

# Define paths
source_path = "/Volumes/workspace/cbl_bronze/raw/Data_Module_1/landing/sfa_app/orders/"
target_table = "workspace.cbl_bronze.sfa_orders"
checkpoint_path = f"/Volumes/workspace/cbl_bronze/raw/_checkpoints/sfa_orders"
schema_location = f"/Volumes/workspace/cbl_bronze/raw/_schemas/sfa_orders"

print(f"Starting Auto Loader ingestion for SFA orders...")
print(f"Source: {source_path}")
print(f"Target: {target_table}")
print(f"Batch ID: {batch_id}\n")

# Read with Auto Loader (cloudFiles)
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")  # JSONL files
    .option("cloudFiles.schemaLocation", schema_location)  # Schema storage location
    .option("cloudFiles.inferColumnTypes", "false")  # No type inference - all STRING
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Schema evolution
    .option("cloudFiles.includeExistingFiles", "true")  # Process existing files
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")  # Capture malformed records
    .load(source_path)
)

# Add lineage metadata columns
df_with_metadata = (df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

# Write to Delta table with Auto Loader streaming
query = (df_with_metadata.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)  # Process available data once and stop
    .toTable(target_table)
)

# Wait for completion
query.awaitTermination()

print(f"\nIngestion completed successfully!")
print(f"Data written to: {target_table}")

In [0]:
import os

dist_path = "/Volumes/workspace/cbl_bronze/raw/Data_Module_1/landing/distributor_master/"

print("Files in distributor_master folder:\n")
for file in sorted(os.listdir(dist_path)):
    file_path = os.path.join(dist_path, file)
    if os.path.isfile(file_path):
        file_size = os.path.getsize(file_path)
        print(f"  {file} ({file_size:,} bytes)")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Define paths
source_file = "/Volumes/workspace/cbl_bronze/raw/Data_Module_1/landing/distributor_master/Distributor_Master_FY25 (final v3).xlsx"
target_table = "workspace.cbl_bronze.distributor_master_raw"

print(f"Starting ingestion for distributor master...")
print(f"Source: {source_file}")
print(f"Target: {target_table}\n")

# Read Excel file using SQL read_files with no header
df = spark.sql(f"""
    SELECT * FROM read_files(
        '{source_file}',
        format => 'excel',
        header => false
    )
""")

print(f"Loaded {df.count():,} rows")
print(f"Columns detected: {len(df.columns)}\n")

# Rename columns from _c0, _c1, ... to col_00, col_01, ...
column_mapping = {col_name: f"col_{i:02d}" for i, col_name in enumerate(df.columns)}
df_renamed = df.select([df[old_name].alias(new_name) for old_name, new_name in column_mapping.items()])

print(f"Renamed columns: {', '.join(df_renamed.columns[:5])}...\n")

# Add metadata columns
df_with_metadata = (df_renamed
    .withColumn("_source_file", lit(source_file))
    .withColumn("_ingest_timestamp", current_timestamp())
)

# Write to bronze table
df_with_metadata.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

row_count = df_with_metadata.count()
print(f"Ingestion completed successfully!")
print(f"Data written to: {target_table}")
print(f"Rows ingested: {row_count:,}")
print(f"Total columns: {len(df_with_metadata.columns)}")

In [0]:
%sql
-- Verify the raw distributor master data
SELECT 
  col_00, col_01, col_02, col_03, col_04, col_05, col_06,
  _source_file,
  _ingest_timestamp
FROM workspace.cbl_bronze.distributor_master_raw
LIMIT 10;

In [0]:
%sql
-- Check total row count and show the last few rows (including TOTAL row)
SELECT 
  'Total Rows' as metric,
  COUNT(*) as value
FROM workspace.cbl_bronze.distributor_master_raw

UNION ALL

SELECT
  'Data Columns' as metric,
  7 as value;  -- col_00 to col_06

-- Show last 5 rows to verify TOTAL row is preserved
SELECT col_00, col_01, col_02, col_03, col_04, col_05, col_06
FROM workspace.cbl_bronze.distributor_master_raw
ORDER BY _ingest_timestamp DESC, col_00 DESC
LIMIT 5;

In [0]:
import os

weather_path = "/Volumes/workspace/cbl_bronze/raw/Data_Module_1/landing/external_api/weather/"

print("Contents of weather folder:\n")
for item in sorted(os.listdir(weather_path)):
    item_path = os.path.join(weather_path, item)
    if os.path.isdir(item_path):
        print(f"  [DIR] {item}")
        # List files in subdirectory
        for subfile in sorted(os.listdir(item_path)):
            subfile_path = os.path.join(item_path, subfile)
            if os.path.isfile(subfile_path):
                file_size = os.path.getsize(subfile_path)
                print(f"    {subfile} ({file_size:,} bytes)")
    elif os.path.isfile(item_path):
        file_size = os.path.getsize(item_path)
        print(f"  {item} ({file_size:,} bytes)")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
import uuid

# Generate a unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())

# Define paths
source_path = "/Volumes/workspace/cbl_bronze/raw/Data_Module_1/landing/external_api/weather/*/*"
target_table = "workspace.cbl_bronze.weather_raw"

print(f"Starting batch ingestion for weather data...")
print(f"Source: {source_path}")
print(f"Target: {target_table}")
print(f"Batch ID: {batch_id}\n")

# Read multiline JSON with no type inference
df = (spark.read
    .format("json")
    .option("multiLine", "true")  # Multiline JSON
    .option("inferSchema", "false")  # No type inference - all STRING
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")  # Capture malformed records
    .load(source_path)
)

print(f"Loaded {df.count():,} rows")
print(f"Columns detected: {len(df.columns)}\n")

# Add lineage metadata columns
df_with_metadata = (df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

# Write to bronze table with append mode
df_with_metadata.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

row_count = df_with_metadata.count()
print(f"Ingestion completed successfully!")
print(f"Data written to: {target_table}")
print(f"Rows ingested: {row_count:,}")
print(f"Total columns: {len(df_with_metadata.columns)}")

In [0]:
%sql
-- Verify weather data with nested structure preserved
SELECT 
  stationId,
  station,
  retrievedAt,
  observations,  -- Nested array preserved
  SIZE(observations) as observation_count,
  _source_file,
  _ingest_timestamp
FROM workspace.cbl_bronze.weather_raw
LIMIT 3;

In [0]:
%sql
-- Summary of weather data ingestion
SELECT 
  'weather_raw' as table_name,
  COUNT(*) as monthly_records,
  COUNT(DISTINCT stationId) as distinct_stations,
  SUM(SIZE(observations)) as total_daily_observations,
  COUNT(DISTINCT _batch_id) as batch_count
FROM workspace.cbl_bronze.weather_raw;

In [0]:
import os

calendar_path = "/Volumes/workspace/cbl_bronze/raw/Data_Module_1/landing/external_api/calendar/"

print("Files in calendar folder:\n")
for file in sorted(os.listdir(calendar_path)):
    file_path = os.path.join(calendar_path, file)
    if os.path.isfile(file_path):
        file_size = os.path.getsize(file_path)
        print(f"  {file} ({file_size:,} bytes)")

In [0]:
%sql
-- Analyze KNA1 customer table
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT KUNNR) as distinct_customers,
  COUNT(CASE WHEN LOEVM = 'X' THEN 1 END) as deleted_customers,
  COUNT(CASE WHEN LOEVM != 'X' OR LOEVM IS NULL THEN 1 END) as active_customers
FROM workspace.cbl_bronze.kna1_customer;

In [0]:
%sql
-- Show examples of duplicate KUNNR values with their details
WITH duplicate_kunnr AS (
  SELECT KUNNR
  FROM workspace.cbl_bronze.kna1_customer
  GROUP BY KUNNR
  HAVING COUNT(*) > 1
  LIMIT 5
)
SELECT 
  c.KUNNR,
  c.NAME1,
  c.ERDAT,
  c.LOEVM
FROM workspace.cbl_bronze.kna1_customer c
INNER JOIN duplicate_kunnr d ON c.KUNNR = d.KUNNR
ORDER BY c.KUNNR, c.ERDAT;

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
import uuid

# Generate a unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())

# Define paths
source_path = "/Volumes/workspace/cbl_bronze/raw/Data_Module_1/landing/external_api/calendar/sl_holiday_calendar_2024_2025.csv"
target_table = "workspace.cbl_bronze.holiday_calendar"

print(f"Starting ingestion for holiday calendar...")
print(f"Source: {source_path}")
print(f"Target: {target_table}")
print(f"Batch ID: {batch_id}\n")

# Read CSV with all columns as string
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # No type inference - all STRING
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")  # Capture malformed records
    .load(source_path)
)

print(f"Loaded {df.count():,} rows")
print(f"Columns: {', '.join(df.columns)}\n")

# Add lineage metadata columns
df_with_metadata = (df
    .withColumn("_source_file", lit(source_path))
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

# Write to bronze table with append mode
df_with_metadata.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

row_count = df_with_metadata.count()
print(f"Ingestion completed successfully!")
print(f"Data written to: {target_table}")
print(f"Rows ingested: {row_count:,}")
print(f"Total columns: {len(df_with_metadata.columns)}")

In [0]:
%sql
-- Verify the holiday calendar data
SELECT 
  cal_date,
  holiday_name,
  public_holiday,
  poya_day,
  liquor_sales_prohibited,
  _source_file,
  _ingest_timestamp
FROM workspace.cbl_bronze.holiday_calendar
ORDER BY cal_date
LIMIT 10;

In [0]:
%sql
-- Check columns in bronze mara_material
DESCRIBE TABLE workspace.cbl_bronze.mara_material;

In [0]:
%sql
-- Check columns in silver fact_billing
DESCRIBE TABLE workspace.cbl_silver.fact_billing;

In [0]:
%sql
-- 1. Sample MATNR values from bronze mara_material
SELECT 'Bronze MARA' as source, MATNR as material_code
FROM workspace.cbl_bronze.mara_material
ORDER BY MATNR
LIMIT 15;

-- 2. Sample distinct material_id values from silver fact_billing
SELECT 'Silver Billing' as source, material_id as material_code
FROM (
  SELECT DISTINCT material_id
  FROM workspace.cbl_silver.fact_billing
  ORDER BY material_id
  LIMIT 15
);

In [0]:
%sql
-- Count summary: total materials, materials with sales, materials without sales
WITH material_sales AS (
  SELECT DISTINCT m.MATNR
  FROM workspace.cbl_bronze.mara_material m
  LEFT JOIN workspace.cbl_silver.fact_billing b
    ON m.MATNR = b.material_id
  WHERE b.material_id IS NOT NULL
),
material_no_sales AS (
  SELECT DISTINCT m.MATNR
  FROM workspace.cbl_bronze.mara_material m
  LEFT JOIN workspace.cbl_silver.fact_billing b
    ON m.MATNR = b.material_id
  WHERE b.material_id IS NULL
)
SELECT 
  'Total materials in MARA' as metric,
  COUNT(DISTINCT MATNR) as count
FROM workspace.cbl_bronze.mara_material

UNION ALL

SELECT 
  'Materials with sales' as metric,
  COUNT(*) as count
FROM material_sales

UNION ALL

SELECT 
  'Materials WITHOUT sales' as metric,
  COUNT(*) as count
FROM material_no_sales

UNION ALL

SELECT 
  'Distinct materials in billing' as metric,
  COUNT(DISTINCT material_id) as count
FROM workspace.cbl_silver.fact_billing;

-- Show sample of materials with no sales
SELECT 
  m.MATNR,
  m.MAKTX as material_description,
  m.ZZBRAND as brand,
  m.ZZCATEGORY as category,
  m.ZZLISTPRICE as list_price
FROM workspace.cbl_bronze.mara_material m
LEFT JOIN workspace.cbl_silver.fact_billing b
  ON m.MATNR = b.material_id
WHERE b.material_id IS NULL
ORDER BY m.MATNR
LIMIT 10;

In [0]:
%sql
-- Summary: Count materials with and without sales
SELECT 
  COUNT(DISTINCT m.MATNR) as total_materials_in_mara,
  COUNT(DISTINCT CASE WHEN b.material_id IS NOT NULL THEN m.MATNR END) as materials_with_sales,
  COUNT(DISTINCT CASE WHEN b.material_id IS NULL THEN m.MATNR END) as materials_without_sales,
  COUNT(DISTINCT b.material_id) as distinct_materials_in_billing
FROM workspace.cbl_bronze.mara_material m
LEFT JOIN workspace.cbl_silver.fact_billing b
  ON m.MATNR = b.material_id;

In [0]:
# CRITICAL VALIDATION: Row count must match expected value
EXPECTED_ROW_COUNT = 249340

query = f"""
SELECT COUNT(*) as row_count
FROM {catalog}.{schema_bronze}.sap_billing
"""

actual_row_count = spark.sql(query).collect()[0]['row_count']

if actual_row_count != EXPECTED_ROW_COUNT:
    error_msg = (
        f"ASSERTION FAILED: Bronze layer row count mismatch!\n"
        f"  Expected: {EXPECTED_ROW_COUNT:,} rows\n"
        f"  Actual:   {actual_row_count:,} rows\n"
        f"  Difference: {actual_row_count - EXPECTED_ROW_COUNT:+,} rows\n\n"
        f"This indicates a data quality issue. Investigation required before proceeding."
    )
    raise AssertionError(error_msg)

print(f"✓ ASSERTION PASSED: Row count is exactly {actual_row_count:,} as expected")